# 27 — One longer natural-loop search in Colab

Notebook 26 found no exact loop in its first 384 tokens. The even-sum prompt finished
thinking; the substring prompt was still thinking and reconsidering its code when cut off.
Only the substring prompt is continued here, freshly generated with the same model revision
and greedy decoding. No easy-prompt rerun, training, adapter, or forced repair.

**Goal:** find an actual exact thinking cycle before evaluating RPMR.
**Input:** the same hand-written substring development question.
**Output:** full raw output, token IDs, thinking-closure status, first detector event, and timing.
**Hypothesis:** a longer prefix may reveal a cycle missed by the short cap.
**Change:** total token cap increases to 2,048. **Reference:** the saved 384-token prefix.
**Measure:** exact-cycle detection, natural thinking closure, cap exhaustion, time and estimated units.
**Support:** an observed repeat event in thinking. **Contradiction:** no exact repeat, whether
thinking finishes or reaches the cap. Neither outcome establishes RPMR accuracy or recovery.

One prompt, detection only, at most **2,048 total new tokens**. Stops early if the detector
finds three identical consecutive blocks of 8, 16 or 32 tokens. It does not detect all semantic
repetition. A single “Wait” is not sufficient. No generated code is executed.

**Budget:** at most **two estimated units**, keeping 20 units reserved.
At 6.77 A100 units/hour this allows about **17.7 minutes** from the first cell's start.
Planning estimate **3–8 minutes**, about **0.34–0.90 units at that historical rate**.
Assumes similar setup and roughly 20 tokens/second; longer-prefix speed is not guaranteed.
Enter your actual current balance and live rate. Earlier runtime allocation time is outside
this notebook's timer. No automatic next run or larger pilot follows.

Upload to a fresh Colab GPU runtime and choose **Run all**. Allow Drive.
Report/checkpoints go to `MyDrive/stop-overthinking/rpmr-loop-search/<unique-run>/report.json`.
It attempts report download and runtime disconnection. If disconnection fails, use
**Runtime → Disconnect and delete runtime**. The watchdog uses an entered rate, not live billing.
**Prepared and syntax-checked; not run in Colab yet.**


## 1. Budget and Drive

The timer starts before setup. Enter the current available units, not the original 50.
This uses the same save/watchdog mechanism as the passed notebooks.


In [ ]:
import sys, time, json, math, threading, subprocess, datetime, uuid
from pathlib import Path
from google.colab import drive, runtime

SESSION_START = time.monotonic()
SMOKE_CAP = 2.0
RESERVE = 20.0
BALANCE = float(input("Current Colab compute units available: "))
RATE = float(input("LIVE usage rate in compute units per hour, from Colab Resources: "))
if not math.isfinite(BALANCE) or not math.isfinite(RATE) or RATE <= 0 or BALANCE <= RESERVE:
    raise ValueError("Enter a finite positive rate and balance above the 20-unit reserve.")
SMOKE_CAP = min(SMOKE_CAP, BALANCE - RESERVE)
drive.mount('/content/drive')
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
OUT = Path('/content/drive/MyDrive/stop-overthinking/rpmr-loop-search') / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
REPORT = dict(status='started', run_id=RUN_ID, model='Qwen/Qwen3.5-2B',
              initial_balance=BALANCE, entered_units_per_hour=RATE,
              estimated_unit_cap=SMOKE_CAP, reserve_units=RESERVE, checks=[],
              scope='one natural exact-loop search; no memory repair or accuracy measurement')
LOCK = threading.RLock()
STOP_WATCHDOG = threading.Event()

def used_units():
    return (time.monotonic() - SESSION_START) / 3600 * RATE

def save_report():
    with LOCK:
        REPORT['wall_seconds'] = round(time.monotonic() - SESSION_START, 3)
        REPORT['estimated_units_used'] = round(used_units(), 4)
        temporary = OUT / 'report.json.tmp'
        temporary.write_text(json.dumps(REPORT, indent=2))
        temporary.replace(OUT / 'report.json')

def guard():
    if used_units() >= max(0, SMOKE_CAP - 0.1):
        raise TimeoutError('Estimated smoke-unit ceiling reached. Stop this session.')

def watchdog():
    while not STOP_WATCHDOG.wait(1):
        if used_units() >= max(0, SMOKE_CAP - 0.1):
            with LOCK:
                REPORT['status'] = 'budget_stopped'
                save_report()
            print('Budget ceiling: disconnecting runtime. Report saved to Drive.', flush=True)
            try:
                runtime.unassign()
            except Exception as exc:
                with LOCK:
                    REPORT['disconnect_error'] = str(exc)
                    save_report()
                print('Disconnect manually: Runtime → Disconnect and delete runtime.', flush=True)
            return

save_report()
threading.Thread(target=watchdog, daemon=True).start()
print(f'Smoke cap {SMOKE_CAP:.2f} units; reserve {RESERVE:.0f}. Results: {OUT}')
guard()


## 2. Load 2B, search once, save, and disconnect

Uses the exact model revision from the passed run. Detects only before the thinking-close
marker; afterward it allows the answer to finish within the total cap. Saves every 64 tokens
so a budget stop retains the latest prefix. No internal memory is modified.


In [ ]:
try:
    guard()
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'transformers==5.18.0','accelerate','safetensors'],check=True,
                   timeout=max(1,(SMOKE_CAP-used_units()-0.1)/RATE*3600))
    guard()
    import torch,transformers
    from transformers import AutoTokenizer
    if not torch.cuda.is_available():
        raise RuntimeError('Choose a Colab GPU runtime first.')
    torch.manual_seed(3407)
    torch.cuda.manual_seed_all(3407)
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    MODEL_ID='Qwen/Qwen3.5-2B'
    REVISION='15852e8c16360a2fea060d615a32b45270f8a8fc'
    REPORT['model_revision']=REVISION
    REPORT['environment']=dict(python=sys.version,torch=torch.__version__,
        transformers=transformers.__version__,gpu=torch.cuda.get_device_name(0),model_dtype=str(dtype),seed=3407)
    REPORT['settings']=dict(max_total_new_tokens=2048,decoding='greedy',periods=[8,16,32],repetitions=3,
                            memory_repair_enabled=False,code_execution=False)
    save_report()
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION)
    loader=getattr(transformers,'AutoModelForMultimodalLM',None)
    if loader is None:
        loader=transformers.AutoModelForImageTextToText
    model=loader.from_pretrained(MODEL_ID,revision=REVISION,dtype=dtype,
                                 device_map={'':0},attn_implementation='sdpa')
    model.eval()
    question='Write a Python function longest_unique_substring_length(text) that returns the length of the longest contiguous substring with no repeated characters. Explain how to handle empty text and repeated characters.'
    prompt=tokenizer.apply_chat_template([dict(role='user',content=question)],
        tokenize=False,add_generation_prompt=True,enable_thinking=True)
    ids=tokenizer(prompt,return_tensors='pt').input_ids.to('cuda')
    guard()
    with torch.no_grad():
        output=model(input_ids=ids,attention_mask=torch.ones_like(ids),use_cache=True,logits_to_keep=1)
    cache=output.past_key_values
    cursor=output.logits[:,-1].argmax(dim=-1,keepdim=True)
    eos=model.generation_config.eos_token_id
    eos_ids=set(eos if isinstance(eos,list) else [eos])
    tokens=[]
    event=None
    in_thinking=True
    stop_reason='total_token_cap'
    def period_for(prefix):
        for p in (8,16,32):
            if len(prefix)>=3*p and prefix[-p:]==prefix[-2*p:-p]==prefix[-3*p:-2*p]:
                return p
        return None
    torch.cuda.synchronize()
    started=time.monotonic()
    with torch.no_grad():
        for step in range(2048):
            guard()
            token=int(cursor.item())
            tokens.append(token)
            text=tokenizer.decode(tokens,skip_special_tokens=False)
            if '</think>' in text:
                in_thinking=False
            period=period_for(tokens) if in_thinking else None
            if period is not None:
                event=dict(at_token=len(tokens),period=period,repeats=3,inside_thinking=True,
                           repeated_block_token_ids=tokens[-period:])
                stop_reason='exact_thinking_repeat_detected'
            elif token in eos_ids:
                stop_reason='eos'
            if len(tokens)%64==0 or event is not None or token in eos_ids:
                with LOCK:
                    REPORT['answer']=dict(question=question,generated_tokens=len(tokens),token_ids=list(tokens),
                        raw_output=text,thinking_closed=not in_thinking,first_repeat_event=event)
                    save_report()
            if event is not None or token in eos_ids:
                break
            output=model(input_ids=cursor,
                attention_mask=torch.ones((1,ids.shape[1]+len(tokens)),device='cuda',dtype=torch.long),
                past_key_values=cache,use_cache=True,logits_to_keep=1)
            if not torch.isfinite(output.logits).all().item():
                raise RuntimeError('Non-finite logits.')
            cache=output.past_key_values
            cursor=output.logits[:,-1].argmax(dim=-1,keepdim=True)
    torch.cuda.synchronize()
    seconds=time.monotonic()-started
    REPORT['answer']=dict(question=question,generated_tokens=len(tokens),token_ids=tokens,
        raw_output=tokenizer.decode(tokens,skip_special_tokens=False),thinking_closed=not in_thinking,
        first_repeat_event=event,stop_reason=stop_reason,decode_seconds=seconds,
        tokens_per_second=len(tokens)/max(seconds,1e-9))
    REPORT['status']='exact_repeat_found' if event is not None else 'no_exact_repeat_found'
    REPORT['interpretation']=('Candidate natural loop found; inspect the text before designing paired recovery.' if event is not None
        else 'No exact thinking cycle found. Inspect the output; this is not proof that semantic repetition never occurs.')
    REPORT['accuracy_measured']=False
    REPORT['repairs_applied']=0
    REPORT['peak_allocated_gpu_memory_gb']=torch.cuda.max_memory_allocated()/1e9
    guard()
except Exception as exc:
    with LOCK:
        if REPORT.get('status')!='budget_stopped':
            REPORT['status']='failed'
        REPORT['error']=f'{type(exc).__name__}: {exc}'
    import traceback
    traceback.print_exc()
finally:
    save_report()
    # Full token IDs remain in the downloadable/Drive report; keep console readable.
    console=dict(REPORT)
    if 'answer' in console:
        console['answer']={k:v for k,v in console['answer'].items() if k!='token_ids'}
    print(json.dumps(console,indent=2))
    print('Drive report:',OUT/'report.json')
    try:
        from google.colab import files
        files.download(str(OUT/'report.json'))
    except Exception as exc:
        print('Download unavailable; report remains on Drive:',exc)
    try:
        runtime.unassign()
    except Exception as exc:
        REPORT['disconnect_error']=str(exc)
        save_report()
        print('Disconnect manually: Runtime → Disconnect and delete runtime.')
    else:
        STOP_WATCHDOG.set()
